# US Road Paving and Repair: Data Notebook

This notebook builds the data behind the road paving project from three
FHWA sources.

| Source | What it gives | Used for |
|------------------------|------------------------|------------------------|
| Highway Statistics, Table SF-12A | State highway agency capital outlay by kind of work | Dollars spent on repair |
| National Highway Construction Cost Index (NHCCI) | Price index for highway construction | Adjusting dollars for inflation |
| Highway Statistics, Table FA-10 | Miles and cost of federal-aid projects by kind of work | Miles repaved, and miles per dollar |

**Two different measures of “repair”**

- SF-12A does not report resurfacing on its own in most years, so
  spending uses FHWA’s whole **3R** group: resurfacing, restoration,
  rehabilitation, and minor widening.
- FA-10 reports **resurfacing** and **restoration and rehabilitation**
  separately, with miles. Those two lines are added together.

**Caveats**

- SF-12A covers what state highway agencies spent. FA-10 covers only
  projects with federal-aid funds authorized that fiscal year. The two
  are not the same set of projects, so miles are only ever divided by
  FA-10’s own costs.
- Local-class roads only appear in SF-12A from 2015, so they are left
  out to keep the years comparable.
- When a state did not report, FHWA reuses its previous year’s figures.
- FHWA published no FA-10 table in its 2021 edition.

In [1]:
# Needs: requests beautifulsoup4 pandas openpyxl xlrd matplotlib
import re, time, sys
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# Shared site chart style (site_style.py in the repo root)
sys.path.insert(0, str(next(p for p in Path.cwd().resolve().parents
                            if (p / "_quarto.yml").exists())))
import site_style as ss
ss.apply()

RAW = Path("data/raw/highway_statistics"); RAW.mkdir(parents=True, exist_ok=True)
PROC = Path("data/processed");             PROC.mkdir(parents=True, exist_ok=True)

FHWA_STATS = "https://www.fhwa.dot.gov/policyinformation/statistics.cfm"
NHCCI_CSV = ("https://datahub.transportation.gov/resource/r94d-n4f9.csv"
             "?$select=quarter,nhcci&$order=quarter&$limit=1000")
TABLES = ("sf12a", "fa10")
FIRST_YEAR = 2012      # first year SF-12A is published in this layout
REFRESH = False        # True: ask FHWA for years and index values not yet downloaded

### Download

Files are downloaded once into `data/raw/` and reused after that. Set
`REFRESH = True` above to pick up a newly published year.

In [2]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "research-notebook/1.0"})


def soup(url):
    r = SESSION.get(url, timeout=60); r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def fetch(url, dest, pause=1.0):
    """Download once, reuse thereafter. Returns the local Path."""
    if dest.exists() and dest.stat().st_size > 0 and not (REFRESH and dest.suffix == ".csv"):
        return dest
    r = SESSION.get(url, timeout=120); r.raise_for_status()
    dest.write_bytes(r.content)
    time.sleep(pause)   # FHWA is a public good; do not hammer it
    return dest


def find_year_pages():
    """{year: url} for each Highway Statistics edition."""
    out = {}
    for opt in soup(FHWA_STATS).find_all("option"):
        m = re.search(r"(19|20)\d{2}", opt.get_text(" ", strip=True))
        val = opt.get("value", "")
        if m and val and val != "#":
            out[int(m.group(0))] = urljoin(FHWA_STATS, val)
    return dict(sorted(out.items()))


def download_tables():
    """Fetch the Excel file for each table in TABLES from every edition since FIRST_YEAR."""
    for year, page in find_year_pages().items():
        if year < FIRST_YEAR:
            continue
        for a in soup(page).find_all("a", href=True):
            stem = Path(a["href"]).stem.lower().replace("-", "")
            if a.get_text(" ", strip=True) == "Excel" and stem in TABLES:
                fetch(urljoin(page, a["href"]), RAW / f"{year}_{stem}{Path(a['href']).suffix}")


def cached(table):
    return sorted(RAW.glob(f"*_{table}.xls*"))


if REFRESH or not all(cached(t) for t in TABLES):
    download_tables()
nhcci_file = fetch(NHCCI_CSV, RAW.parent / "nhcci_quarterly.csv")

for t in TABLES:
    files = cached(t)
    print(f"{t}: {len(files)} workbooks, {files[0].name[:4]} to {files[-1].name[:4]}")

sf12a: 13 workbooks, 2012 to 2024
fa10: 12 workbooks, 2012 to 2024

### Spending: read every sheet of SF-12A

Each sheet has the states down the side and the improvement types across
the top, for one road class in rural or urban areas. Headings are
matched by name, and an unrecognised heading stops the run, so a change
in FHWA’s layout cannot slip through as a wrong number.

In [3]:
# Canonical improvement types, keyed by the bottom two header rows of each column.
IMPROVEMENT_TYPES = {
    "ROW": ("Right of way", "Right of way and engineering"),
    "ENGINEERING": ("Engineering", "Right of way and engineering"),
    "CONSTRUC- TION": ("New construction", "New roads and added capacity"),
    "RELOCATION": ("Relocation", "New roads and added capacity"),
    "ADDED CAPACITY": ("Reconstruction, added capacity", "New roads and added capacity"),
    "MAJOR WIDENING": ("Major widening", "New roads and added capacity"),
    "NO ADDED CAPACITY": ("Reconstruction, no added capacity", "Reconstruction, no added capacity"),
    "MINOR WIDENING": ("Minor widening", "Repaving and repair (3R)"),
    "RESTORATION & REHABILITATION": ("Restoration and rehabilitation", "Repaving and repair (3R)"),
    "RESURFACING": ("Resurfacing", "Repaving and repair (3R)"),
    "RESTORATION & REHAB. & RESURF.": ("Restoration, rehabilitation and resurfacing", "Repaving and repair (3R)"),
    "NEW BRIDGE": ("New bridge", "Bridges"),
    "BRIDGE REPLACEMENT": ("Bridge replacement", "Bridges"),
    "MAJOR BRIDGE REHAB.": ("Major bridge rehabilitation", "Bridges"),
    "MINOR BRIDGE REHAB.": ("Minor bridge rehabilitation", "Bridges"),
    "MAJOR BRDG REHAB.": ("Major bridge rehabilitation", "Bridges"),
    "MINOR BRDG REHAB.": ("Minor bridge rehabilitation", "Bridges"),
    "SAFETY": ("Safety", "Safety, traffic and other"),
    "CONTROL SYSTEMS": ("Traffic operations and control", "Safety, traffic and other"),
    "ENHANCEMT & OTHER": ("Environmental enhancement and other", "Safety, traffic and other"),
}
THREE_R = "Repaving and repair (3R)"


def squash(value):
    """Collapse whitespace; blank for empty cells."""
    return "" if pd.isna(value) else " ".join(str(value).split())


def read_sheet(raw, year, sheet):
    """One SF-12A sheet -> tidy rows, plus the sheet's own Total row for checking.

    Each sheet covers one road class in rural or urban areas. Rows run from
    Alabama down to the row before "Total"; the two rows above Alabama name
    each column's improvement type.
    """
    first_col = raw.iloc[:, 0].map(squash)
    start = first_col[first_col.str.upper() == "ALABAMA"].index[0]
    total = first_col[first_col.str.fullmatch("(?i)total")].index[0]

    title = next(t for t in raw.iloc[:start].stack().map(squash)
                 if re.search(r"\b(RURAL|URBAN)\b", t.upper()) and "TABLE" not in t.upper())
    title = re.sub(r"^FOR\s+", "", title.upper())
    area = "Urban" if "URBAN" in title else "Rural"
    road_class = (title.replace("SYSTEMS", "").replace("SYSTEM", "")
                  .replace("IN RURAL AREAS", "").replace("IN URBAN AREAS", ""))
    road_class = " ".join(road_class.split()).capitalize()

    labels = (raw.iloc[start - 2].map(squash) + " " + raw.iloc[start - 1].map(squash)).str.strip()
    rows, sheet_total = [], {}
    for col, label in labels.items():
        if col == 0 or label in ("", "TOTAL"):
            continue
        if label not in IMPROVEMENT_TYPES:
            raise ValueError(f"{year} {sheet}: unrecognised column heading {label!r}")
        improvement, group = IMPROVEMENT_TYPES[label]
        values = pd.to_numeric(raw.loc[start:total - 1, col], errors="coerce")
        states = first_col.loc[start:total - 1]
        keep = states != ""
        rows.append(pd.DataFrame({
            "year": year, "area": area, "road_class": road_class,
            # footnote marks such as "Utah (2)" or "Maine 2/" come off the name
            "state": states[keep].str.replace(r"(\s*(\(\d+\)|\d+/))+$", "", regex=True),
            "improvement": improvement, "group": group,
            "dollars_thousands": values[keep].fillna(0.0),
        }))
        sheet_total[improvement] = pd.to_numeric(raw.at[total, col], errors="coerce")
    return pd.concat(rows, ignore_index=True), sheet_total


def read_workbook(path):
    """Every sheet of one year's SF-12A workbook."""
    year = int(path.name[:4])
    sheets = pd.read_excel(path, sheet_name=None, header=None, dtype=object)
    frames, mismatches = [], []
    for sheet, raw in sheets.items():
        tidy_sheet, sheet_total = read_sheet(raw, year, sheet)
        frames.append(tidy_sheet)
        ours = tidy_sheet.groupby("improvement")["dollars_thousands"].sum()
        for improvement, published in sheet_total.items():
            if abs(ours[improvement] - published) > 1:   # thousands of dollars
                mismatches.append((year, sheet, improvement, ours[improvement], published))
    return pd.concat(frames, ignore_index=True), mismatches

In [4]:
frames, mismatches = [], []
for path in cached("sf12a"):
    tidy_year, bad = read_workbook(path)
    frames.append(tidy_year)
    mismatches += bad

tidy = pd.concat(frames, ignore_index=True)

# Every sheet's state rows must add up to the Total row FHWA prints on it.
if mismatches:
    raise ValueError(f"{len(mismatches)} column totals do not match FHWA's: {mismatches[:5]}")

print(f"{len(tidy):,} rows from {len(frames)} workbooks; "
      f"every column total matches the published Total row.")

134,844 rows from 13 workbooks; every column total matches the published Total row.

### Inflation: the highway construction cost index

NHCCI is quarterly. Each year’s index is the average of its four
quarters, and dollars are restated in the prices of the latest year of
spending data.

In [5]:
quarterly = pd.read_csv(nhcci_file)
quarterly["year"] = quarterly["quarter"].str[:4].astype(int)
by_year = quarterly.groupby("year")["nhcci"].agg(["mean", "size"])
nhcci = by_year.loc[by_year["size"] == 4, "mean"]     # complete years only

comparable = tidy[tidy["road_class"] != "Local"]
BASE_YEAR = int(comparable["year"].max())
to_base_dollars = nhcci[BASE_YEAR] / nhcci             # multiply nominal dollars by this

print(f"Construction prices rose {nhcci[BASE_YEAR] / nhcci[FIRST_YEAR] - 1:.0%} "
      f"from {FIRST_YEAR} to {BASE_YEAR}. Dollars below are in {BASE_YEAR} prices.")

Construction prices rose 102% from 2012 to 2024. Dollars below are in 2024 prices.

### National spending series

Local-class roads are dropped here so that every year covers the same
road classes: Interstates, other arterials, and collectors.

In [6]:
national = comparable.groupby(["year", "group"], as_index=False)["dollars_thousands"].sum()
national.to_csv(PROC / "national_outlay_by_group.csv", index=False)

by_state = (comparable.assign(three_r=comparable["dollars_thousands"].where(comparable["group"] == THREE_R, 0.0))
            .groupby(["year", "state"], as_index=False)
            .agg(three_r_thousands=("three_r", "sum"), total_thousands=("dollars_thousands", "sum")))
by_state.to_csv(PROC / "state_3r_outlay.csv", index=False)

billions = national.pivot(index="year", columns="group", values="dollars_thousands") / 1e6
summary = pd.DataFrame({
    "three_r_bn": billions[THREE_R],
    "all_capital_bn": billions.sum(axis=1),
})
summary["three_r_share"] = summary["three_r_bn"] / summary["all_capital_bn"]
summary["nhcci"] = nhcci.reindex(summary.index)
summary["three_r_real_bn"] = summary["three_r_bn"] * to_base_dollars.reindex(summary.index)
summary["base_year"] = BASE_YEAR
summary.to_csv(PROC / "national_3r_summary.csv")

(summary.drop(columns="base_year")
 .rename(columns={"three_r_bn": "3R outlay ($bn)", "all_capital_bn": "All capital outlay ($bn)",
                  "three_r_share": "3R share", "nhcci": "NHCCI",
                  "three_r_real_bn": f"3R outlay ({BASE_YEAR} $bn)"})
 .style.format({"3R outlay ($bn)": "{:.1f}", "All capital outlay ($bn)": "{:.1f}",
                "3R share": "{:.0%}", "NHCCI": "{:.2f}",
                f"3R outlay ({BASE_YEAR} $bn)": "{:.1f}"}))

In [7]:
years = summary.index
share = summary["three_r_share"] * 100
nominal = summary["three_r_bn"]
real = summary["three_r_real_bn"]
first, last = years[0], years[-1]
change = real[last] / real[first] - 1

fig, (ax, ax_d) = plt.subplots(
    2, 1, figsize=(9, 6.8), sharex=True,
    gridspec_kw={"height_ratios": [1.7, 2.1], "hspace": 0.3},
)

ax.plot(years, share, color=ss.BRICK, marker="o", markersize=4)
ax.set_ylim(0, 40)
ax.set_yticks([0, 10, 20, 30, 40], ["0", "10", "20", "30", "40%"])
for y in (first, last):
    ss.callout(ax, y, share[y], f"{share[y]:.0f}%", dy=10)
ss.axis_heading(ax, "Share of state highway capital outlay")

ax_d.bar(years, real, width=0.7, color=ss.BRICK, alpha=0.38)
ax_d.plot(years, nominal, color=ss.INDIGO, marker="o", markersize=3.5)
for y in (first, last):
    ss.callout(ax_d, y, real[y], f"${real[y]:.0f}bn", dy=5)
ss.end_label(ax_d, last, nominal[last], "As spent", color=ss.INDIGO, dx=18, dy=-10)
ax_d.set_ylim(0, real.max() * 1.22)
ticks = list(range(0, int(real.max()) + 1, 10))
ax_d.set_yticks(ticks, ["$0"] + [f"${t}bn" for t in ticks[1:]])
ss.axis_heading(ax_d, f"Dollars spent, in {BASE_YEAR} construction prices (bars)")
ax_d.set_xticks(list(years))
ax_d.set_xlim(first - 0.7, last + 0.7)

ss.titles(fig, f"After inflation, states spent {abs(change):.0%} "
               f"{'less' if change < 0 else 'more'} on road repair in {last} than in {first}",
          f"State highway agency spending on 3R work, {first} to {last}", y=0.93)
ss.source_note(fig, "Source: FHWA Highway Statistics, Table SF-12A; FHWA National Highway "
                    "Construction Cost Index. Excludes local-class roads.", y=0.04)
plt.show()

In [8]:
latest = billions.index[-1]
mix = billions.loc[latest].sort_values()
colors = [ss.BRICK if group == THREE_R else ss.FAINT for group in mix.index]

fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.barh(mix.index, mix.values, color=colors, height=0.66)
for group, value in mix.items():
    ax.text(value + 0.3, group, f"${value:.1f}bn", va="center", fontsize=10.5, color=ss.MUTED)
ax.grid(False)
ax.spines["bottom"].set_visible(False)
ax.set_xticks([])
ax.set_xlim(0, mix.max() * 1.15)
ax.tick_params(axis="y", labelcolor=ss.INK)

ss.titles(fig, "Repair is the largest single use of highway capital money",
          f"State highway agency capital outlay by kind of work, {latest}", y=0.93)
ss.source_note(fig, "Source: FHWA Highway Statistics, Table SF-12A. Excludes local-class roads.",
               y=0.0)
plt.show()

### Miles: Table FA-10

FA-10 lists, for projects with federal-aid funds authorized in each
fiscal year, the miles covered and the total project cost by kind of
work. Miles here are divided only by FA-10’s own costs, restated in the
base year’s prices.

The split between “resurfacing” and “restoration and rehabilitation”
moves around a lot from year to year, which looks like reclassification,
so the two are added together.

In [9]:
FA10_MEASURES = {"number of miles": "miles", "total cost": "total_cost_thousands",
                 "federal funds": "federal_funds_thousands"}


def read_fa10(path):
    """One year's FA-10 table -> rows of (year, improvement, measure, value).

    The table stacks a block per improvement type: a heading row, then
    "Number Of Miles", "Federal Funds" and "Total Cost" rows across the road
    classes, ending in a total column. Only the total column is kept, after
    checking that it equals the sum of the road classes.
    """
    year = int(path.name[:4])
    raw = pd.read_excel(path, header=None, dtype=object)
    text = raw.map(squash)
    measure_col = next(c for c in text.columns
                       if (text[c].str.lower() == "number of miles").any())
    rows, improvement = [], None
    for i in raw.index:
        heading = " ".join(t for t in text.loc[i, :measure_col - 1] if t)
        measure = text.at[i, measure_col].lower()
        if heading:
            improvement = re.sub(r"\s*(\(\d+\)|\d+/)$", "", heading)
        if improvement == "U.S. Total":
            continue   # not used, and FHWA's own grand-total row does not always add up
        if measure in FA10_MEASURES and improvement:
            values = pd.to_numeric(raw.loc[i, measure_col + 1:], errors="coerce").dropna()
            total, parts = values.iloc[-1], values.iloc[:-1].sum()
            if abs(total - parts) > max(1.0, abs(total) * 0.001):
                raise ValueError(f"{year} FA-10 {improvement} {measure}: "
                                 f"total {total} is not the sum of its columns {parts}")
            rows.append({"year": year, "improvement": improvement,
                         "measure": FA10_MEASURES[measure], "value": float(total)})
    return pd.DataFrame(rows)

In [10]:
fa10 = pd.concat([read_fa10(p) for p in cached("fa10")], ignore_index=True)

REPAVING = ["Resurfacing", "Restoration & Rehabilitation"]
repaving = (fa10[fa10["improvement"].isin(REPAVING)]
            .pivot_table(index="year", columns="measure", values="value", aggfunc="sum"))

miles = pd.DataFrame({
    "miles": repaving["miles"],
    "total_cost_bn": repaving["total_cost_thousands"] / 1e6,
})
miles["total_cost_real_bn"] = miles["total_cost_bn"] * to_base_dollars.reindex(miles.index)
miles["miles_per_million_real"] = miles["miles"] / (miles["total_cost_real_bn"] * 1000)
miles["base_year"] = BASE_YEAR
miles.to_csv(PROC / "federal_aid_repaving_miles.csv")

missing = sorted(set(range(miles.index.min(), miles.index.max() + 1)) - set(miles.index))
print(f"No FA-10 table for: {missing or 'none'}")
(miles.drop(columns="base_year")
 .rename(columns={"miles": "Miles", "total_cost_bn": "Total cost ($bn)",
                  "total_cost_real_bn": f"Total cost ({BASE_YEAR} $bn)",
                  "miles_per_million_real": f"Miles per $1m ({BASE_YEAR} $)"})
 .style.format({"Miles": "{:,.0f}", "Total cost ($bn)": "{:.1f}",
                f"Total cost ({BASE_YEAR} $bn)": "{:.1f}",
                f"Miles per $1m ({BASE_YEAR} $)": "{:.2f}"}))

No FA-10 table for: [2021]

In [11]:
# Reindex so a year without a table shows as a gap, not a straight line across it.
span = range(miles.index.min(), miles.index.max() + 1)
m = miles.reindex(span)
thousand_miles = m["miles"] / 1000
per_million = m["miles_per_million_real"]
first, last = miles.index[0], miles.index[-1]

fig, (ax, ax_r) = plt.subplots(
    2, 1, figsize=(9, 6.8), sharex=True,
    gridspec_kw={"height_ratios": [2, 2], "hspace": 0.3},
)

ax.bar(m.index, thousand_miles, width=0.7, color=ss.BRICK, alpha=0.38)
for y in (first, last):
    ss.callout(ax, y, thousand_miles[y], f"{miles.at[y, 'miles']:,.0f}", dy=5)
ax.set_ylim(0, thousand_miles.max() * 1.2)
ax.set_yticks([0, 20, 40, 60], ["0", "20k", "40k", "60k"])
ss.axis_heading(ax, "Miles of road in projects authorized each year")

ax_r.plot(m.index, per_million, color=ss.BRICK, marker="o", markersize=4)
for y in (first, last):
    ss.callout(ax_r, y, per_million[y], f"{per_million[y]:.1f}", dy=10)
ax_r.set_ylim(0, per_million.max() * 1.3)
ss.axis_heading(ax_r, f"Miles per $1 million of project cost, in {BASE_YEAR} construction prices")
for y in missing:
    ax_r.text(y, per_million.max() * 0.08, "no table", ha="center", fontsize=9, color=ss.FAINT)
ax_r.set_xticks(list(m.index))
ax_r.set_xlim(first - 0.7, last + 0.7)

ss.titles(fig, f"A million dollars covered {per_million[first]:.1f} miles of repaving in {first}, "
               f"and {per_million[last]:.1f} in {last}",
          "Federal-aid resurfacing, restoration and rehabilitation projects, by fiscal year", y=0.93)
ss.source_note(fig, "Source: FHWA Highway Statistics, Table FA-10; FHWA National Highway "
                    "Construction Cost Index.", y=0.04)
plt.show()

### Reading miles per dollar with care

A fall in miles per dollar is not only a price effect, since prices are
already taken out. It can also mean the work itself changed: more
rehabilitation and less thin overlay, more urban and multi-lane roads,
or more added work such as safety and drainage inside the same project.
FA-10 counts centerline miles of projects, not lane-miles, and cannot
separate these.